<a href="https://colab.research.google.com/github/milicaclm/transportni_problem/blob/main/transportni_problem_projekat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
kapacitet = [20, 30, 20, 10]
potražnja = [10, 40, 30]

cene = np.array([
    [10, 12, 0],
    [8, 4, 3],
    [6, 9, 4],
    [7, 8, 5]
])

cost_df = pd.DataFrame(cene, columns=['D1', 'D2', 'D3'], index=['F1', 'F2', 'F3', 'F4'])

print("Kapaciteti (Supply):", kapacitet)
print("Zahtevi (Demand):", potražnja)
print("Troškovi transporta (Costs):")
print(cost_df)

Kapaciteti (Supply): [20, 30, 20, 10]
Zahtevi (Demand): [10, 40, 30]
Troškovi transporta (Costs):
    D1  D2  D3
F1  10  12   0
F2   8   4   3
F3   6   9   4
F4   7   8   5


In [ ]:
def severozapadni_ugao(kapacitet, potražnja):

    resenje = np.zeros((len(kapacitet), len(potražnja)))

    kap = kapacitet.copy()
    pot = potražnja.copy()

    i = 0
    j = 0

    while i < len(kapacitet) and j < len(potražnja):
        if kap[i] <= pot[j]:
            resenje[i][j] = kap[i]
            pot[j] -= kap[i]
            i += 1
        else:
            resenje[i][j] = pot[j]
            kap[i] -= pot[j]
            j += 1

    return resenje

In [ ]:
def metod_najmanjih_cena(kapacitet, potražnja, cene):

    resenje = np.zeros((len(kapacitet), len(potražnja)))

    kap = kapacitet.copy()
    pot = potražnja.copy()

    cost_indices = [(cene[i][j], i, j) for i in range(len(kapacitet)) for j in range(len(potražnja))]
    cost_indices.sort()

    for cost, i, j in cost_indices:
        if kap[i] == 0 or pot[j] == 0:
            continue
        allocation = min(kap[i], pot[j])
        resenje[i][j] = allocation
        kap[i] -= allocation
        pot[j] -= allocation

    return resenje

In [ ]:
def fogelov_metod(kapacitet, potražnja, costs):

    resenje = np.zeros((len(kapacitet), len(potražnja)))

    kap = kapacitet.copy()
    pot = potražnja.copy()

    INF = int(1e10)

    while np.sum(kap) > 0 and np.sum(pot) > 0:

        redovni_penali = []
        for i in range(len(kap)):
            if kap[i] == 0:
                redovni_penali.append(-INF)
            else:
                red = sorted([costs[i][j] for j in range(len(pot)) if pot[j] > 0])
                if len(red) > 1:
                    redovni_penali.append(red[1] - red[0])
                else:
                    redovni_penali.append(red[0] if len(red) > 0 else -INF)

        kolonski_penali = []
        for j in range(len(pot)):
            if pot[j] == 0:
                kolonski_penali.append(-INF)
            else:
                kolona = sorted([costs[i][j] for i in range(len(kap)) if kap[i] > 0])
                if len(kolona) > 1:
                    kolonski_penali.append(kolona[1] - kolona[0])
                else:
                    kolonski_penali.append(kolona[0] if len(kolona) > 0 else -INF)

        max_red_penal = max(redovni_penali)
        max_kol_penal = max(kolonski_penali)

        if max_red_penal > max_kol_penal:
            i = redovni_penali.index(max_red_penal)
            j = np.argmin([costs[i][j] if pot[j] > 0 else INF for j in range(len(pot))])
        else:
            j = kolonski_penali.index(max_kol_penal)
            i = np.argmin([costs[i][j] if kap[i] > 0 else INF for i in range(len(kap))])

        allocation = min(kap[i], pot[j])
        resenje[i][j] = allocation
        kap[i] -= allocation
        pot[j] -= allocation

        if kap[i] == 0:
            for k in range(len(pot)):
                costs[i][k] = INF
        if pot[j] == 0:
            for k in range(len(kap)):
                costs[k][j] = INF

    return resenje

In [ ]:
resenje = severozapadni_ugao(kapacitet, potražnja)
resenje_df = pd.DataFrame(resenje, columns=['D1', 'D2', 'D3'], index=['F1', 'F2', 'F3', 'F4'])

print("Inicijalno rešenje (Severozapadni ugao):")
print(resenje_df)

resenje = metod_najmanjih_cena(kapacitet, potražnja, cene)
resenje_df = pd.DataFrame(resenje, columns=['D1', 'D2', 'D3'], index=['F1', 'F2', 'F3', 'F4'])

print("Inicijalno rešenje (Metod najmanjih cena):")
print(resenje_df)

resenje = fogelov_metod(kapacitet, potražnja, cene)
resenje_df = pd.DataFrame(resenje, columns=['D1', 'D2', 'D3'], index=['F1', 'F2', 'F3', 'F4'])

print("Inicijalno rešenje (Fogelov metod):")
print(resenje_df)

Inicijalno rešenje (Severozapadni ugao):
      D1    D2    D3
F1  10.0  10.0   0.0
F2   0.0  30.0   0.0
F3   0.0   0.0  20.0
F4   0.0   0.0  10.0
Inicijalno rešenje (Metod najmanjih cena):
      D1    D2    D3
F1   0.0   0.0  20.0
F2   0.0  20.0  10.0
F3  10.0  10.0   0.0
F4   0.0  10.0   0.0
Inicijalno rešenje (Fogelov metod):
      D1    D2    D3
F1   0.0   0.0  20.0
F2   0.0  30.0   0.0
F3  10.0   0.0  10.0
F4   0.0  10.0   0.0


In [ ]:
def iterativni_postupak(cene: np.ndarray, inicijalno_resenje: DataFrame):
  pass
  u = []*inicijalno_resenje
